In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder,OrdinalEncoder,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import confusion_matrix,accuracy_score,classification_report,ConfusionMatrixDisplay,roc_auc_score,roc_curve
import matplotlib.pyplot as plt

# Loading the dataset

In [ ]:
credit_df = pd.read_csv("german_credit_data.csv")
credit_df.head(15)

# Handling the missing values

In [ ]:
credit_df = credit_df.fillna("none")

# Encoding the target variable

In [ ]:
label_enc = LabelEncoder()
credit_df["risk_encoded"] = label_enc.fit_transform(credit_df["Risk"])
credit_df.drop(columns="Risk",inplace=True)

In [ ]:
X = credit_df.drop("risk_encoded",axis=1)
y = credit_df["risk_encoded"]

# Encoding the input features

In [ ]:
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.to_list()
if "Job" in numeric_features:
    numeric_features.remove("Job")
print(numeric_features)


# Dividing the categorical features into nominal and ordinal features

In [ ]:
ordinal_features = ["Saving accounts","Checking account","Job"]
nominal_features = [col for col in categorical_features if col not in ordinal_features]
print(ordinal_features,nominal_features)

# Defining the order of the categories

In [ ]:
saving_order = ["none","little","moderate","quite rich","rich"]
checking_order = ["none","little","moderate","rich"]
job_order = [0,1,2,3]

explicit_cat = [saving_order,checking_order,job_order]

preprocessor = ColumnTransformer(
    transformers=[("num","passthrough",numeric_features),
                  ("ord",OrdinalEncoder(categories=explicit_cat,handle_unknown="use_encoded_value",unknown_value=-1),ordinal_features),
                  ("nom",OneHotEncoder(handle_unknown="ignore"),nominal_features)]
)

# Splitting the dataset into training and testing sets

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=y,random_state=42)

In [ ]:
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_encoded,columns=feature_names)
X_train_df.head()

# Performing feature selection from the encoded features

In [ ]:
baseline_rf = RandomForestClassifier(n_estimators=50,random_state=42,class_weight="balanced")
selector = SelectFromModel(baseline_rf,threshold="median")
X_train_selected = selector.fit_transform(X_train_encoded,y_train)
X_test_selected = selector.transform(X_test_encoded)
print(f"Training dataset shape:{X_train_selected.shape}")
print(f"Test data set shape: {X_test_selected.shape}")

In [ ]:
selected_features = feature_names[selector.get_support()]
print(selected_features)

In [ ]:
final_rf = RandomForestClassifier(n_estimators=50,random_state=42,class_weight="balanced")
final_rf.fit(X_train_selected,y_train)

# Evaluating the model on the test dataset

In [ ]:
y_pred = final_rf.predict(X_test_selected)
print(y_pred)

# Creating the confusion matrix

In [ ]:
conf_mat = confusion_matrix(y_test,y_pred)
print(conf_mat)

# Plotting the confusion matrix

In [ ]:
disp = ConfusionMatrixDisplay(conf_mat,display_labels=["good","bad"])
disp.plot(cmap=plt.cm.plasma)
plt.title("Confusion matrix for random forest algorithm")
plt.show()

In [ ]:
y_probs = final_rf.predict_proba(X_test_selected)[:,1]
model_accuracy = accuracy_score(y_test,y_pred)
auc_score = roc_auc_score(y_test,y_probs)
print("Accuracy: ",model_accuracy)
print("ROC - AUC score",auc_score)

# Plotting the `ROC` curve

In [ ]:
fpr,tpr,thresholds = roc_curve(y_test,y_probs)

In [ ]:
plt.figure(figsize=(8,6))
plt.plot(fpr,tpr,color="darkorange",linewidth=2,label=f"Random forest (AUC = {auc_score:.3f})")
plt.plot([0,1],[0,1],color="navy",linestyle="dashed",label=f"Random gues (AUC: 0.5)")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve")
plt.show()